In [ ]:
import keras
import tensorflow as tf
from os import listdir
import os
import numpy as np
from PIL import Image 
import tomllib
from pathlib import Path



In [ ]:
def load_image_and_mask(image_path, mask_path, image_size=(1024, 1024)):
    # Load and preprocess image
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, image_size)
    image = tf.cast(image, tf.float32) / 255.0

    # Load and preprocess mask 
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.resize(mask, (256, 256), method='nearest')  
    mask = tf.cast(mask, tf.uint8)
    filename = tf.strings.split(image_path, os.sep)[-1]
    return image, mask, filename

with open("config.toml", "rb") as f:
    config = tomllib.load(f)
path_to_dataset = Path(config["paths"]["data"])
image_dir = path_to_dataset / "images"
mask_dir = path_to_dataset / "sam"


# Get sorted lists of image and mask paths (filenames must match)
image_paths = sorted([os.path.join(image_dir, f) for f in os.listdir(image_dir) ])
mask_paths = sorted([os.path.join(mask_dir, f) for f in os.listdir(mask_dir)])


dataset = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
dataset = dataset.map(lambda img, msk: load_image_and_mask(img, msk), num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.batch(8).prefetch(tf.data.AUTOTUNE)

# Example: iterate through the dataset
for images, masks,filename in dataset.take(2):
    print("Images batch shape:", images.shape)
    print("Masks batch shape:", masks.shape)
    print("Filenames:", filename.numpy())
    print("Unique mask values:", tf.unique(tf.reshape(masks, [-1]))[0].numpy())

Sample random points from mask to use as prompt for SAM

In [ ]:
import cv2 as cv
import matplotlib.pyplot as plt
import glob

def get_blob_centers(img, min_area=100):
    # read image through command line
    # img = cv.imread("/home/research/Documents/eco-seg/example_dataset/sam/S14400_D061020_071520_48_I561_NE_L6.png")
    
    # convert the image to grayscale
    gray_image = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    
    # convert the grayscale image to binary image
    ret, thresh = cv.threshold(gray_image,127,255,0)
    
    # find contours in the binary image
    contours, hierarchy = cv.findContours(thresh,cv.RETR_EXTERNAL,cv.CHAIN_APPROX_NONE)

    for c in contours:
        if cv.contourArea(c) < min_area:
            continue
        
        # calculate moments for each contour
        M = cv.moments(c)

        

        # calculate x,y coordinate of center

        if M["m00"] != 0:
            cX = int(M["m10"] / M["m00"])
            cY = int(M["m01"] / M["m00"])
        else:
            cX, cY = 0, 0
        cv.circle(img, (cX, cY), 10, (100, 150, 255), -1)
        cv.putText(img, "centroid", (cX - 25, cY - 25),cv.FONT_HERSHEY_SIMPLEX, 0.5, (100, 150, 255), 2)

        # display the image
        plt.imshow(img)

imgs_path = glob.glob("/home/research/Documents/eco-seg/example_dataset/sam/*.png")

for img_path in imgs_path:
    img = cv.imread(img_path)
    get_blob_centers(img, min_area=1000)
    plt.show()
    


In [ ]:
def add_sam_dummy_inputs(image, mask, filename):
    # image: (1024, 1024, 3)
    # mask: (1024, 1024, 1)
    # filename: string
    batch_size = tf.shape(image)[0]
    return {
        'images': image,
        # 'points': tf.zeros((batch_size, 0, 2), dtype=tf.float32),
        # 'labels': tf.zeros((batch_size, 0), dtype=tf.float32),
        'boxes': tf.zeros((batch_size, 0, 2, 2), dtype=tf.float32),
        # 'masks': tf.zeros((batch_size, 0, 0, 0, 1), dtype=tf.float32),
    }, mask

dataset = dataset.map(add_sam_dummy_inputs, num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
import keras_hub
base_model = keras_hub.models.SAMImageSegmenter.from_preset('sam_base_sa1b')
base_model.trainable = False



In [ ]:
inputs = base_model.input
outputs = base_model(inputs) # use the output of the base model as the input of the new model


In [ ]:
# Add one additional Conv2D layer and an output layer for water segmentation
if isinstance(outputs, dict):
    masks = outputs["masks"]
elif isinstance(outputs, (list, tuple)):
    _, masks = outputs
else:
    raise ValueError("Unexpected output type from base_model")

# Select the first mask (assuming it's the relevant one for water)
mask = masks[:, 0, :, :]  
#print(mask.shape)
mask = keras.layers.Reshape((256, 256, 1))(mask)

# Additional trainable layer
x = keras.layers.Conv2D(64, kernel_size=(3, 3), padding='same', activation='relu')(mask)
output = keras.layers.Conv2D(1, kernel_size=(1, 1), activation='sigmoid')(x)

model = keras.Model(inputs=inputs, outputs=output)
model.summary()

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
                loss=keras.losses.BinaryCrossentropy(),
                metrics=[keras.metrics.BinaryAccuracy()])


In [ ]:
# Train the model
model.fit(dataset, epochs=100, verbose=1)


In [ ]:
# Save the model
model.save("water_segmentation_model.keras")


In [ ]:
# Load the trained model
loaded_model = keras.models.load_model("water_segmentation_model.keras")

# Select a test image path (e.g., the first image in image_paths)
test_image_path = image_paths[0]

# Load and preprocess the test image (reuse load_image_and_mask, but ignore mask)
test_image, _, _ = load_image_and_mask(test_image_path, mask_paths[0])
test_image = tf.expand_dims(test_image, axis=0)  # Add batch dimension

# Prepare dummy SAM inputs as required by the model
dummy_points = tf.zeros((1, 0, 2), dtype=tf.float32)
dummy_labels = tf.zeros((1, 0), dtype=tf.float32)
dummy_boxes = tf.zeros((1, 0, 2, 2), dtype=tf.float32)
dummy_masks = tf.zeros((1, 0, 0, 0, 1), dtype=tf.float32)

sam_inputs = {
    'images': test_image,
    # 'points': dummy_points,
    # 'labels': dummy_labels,
    'boxes': dummy_boxes
    # 'masks': dummy_masks,
}

# Run inference
pred_mask = loaded_model.predict(sam_inputs)
print("Predicted mask shape:", pred_mask.shape)

In [ ]:
import matplotlib.pyplot as plt

pred_mask_2 = (pred_mask > 0.6).astype(np.float32)

# temp = pred_mask

# pred_mask = (pred_mask > 0).astype(np.float32)

# Display the predicted mask for the test image
plt.imshow(pred_mask_2[0, :, :, 0], cmap='gray')
plt.title("Predicted Mask")
plt.axis('off')
plt.show()

print(pred_mask)



In [ ]:
plt.imshow(test_image[0])